# Step 0: llm-compressor 量化通路总览（三 lab 共享）

**目标**：在动手 FP8/AWQ/SmoothQuant 之前，先建立**整条量化通路**的地图——`oneshot(model, dataset, recipe)` 内部到底干了什么、`dataset`/`recipe`/`modifier`/`scheme`/`quantization_config` 这五个组件各自是什么、三种方法是什么关系。s1/s2/s3 都走同一条通路，区别只在 recipe 里塞了什么。

**对应 OUTLINE**：模块 2 导览（本 step 是教学法前置，非 OUTLINE 单独课时）。

> 本 step 是**概念导览**（无量化填空、无 L1/L2/L3 量化验证），以 markdown 讲解为主，配一段最小 oneshot 演示佐证通路。

In [ ]:
%%capture
import pathlib, json
import torch
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

In [ ]:
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml 的目录）；请在模块目录内启动 jupyter")

MODULE_ROOT = _find_module_root(pathlib.Path.cwd())
OUT_ROOT    = MODULE_ROOT / "out"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)

## 1. oneshot 通路：一次调用里发生了什么

三个 lab 的入口都是同一个函数：

```python
oneshot(model, dataset, recipe)
```

它内部是一条流水线（**记住这张数据流图，三个 lab 都套它**）：

```
model(FP 原始) + dataset(校准文本) + recipe([modifier, ...])
        │
        ▼  oneshot(...)
   ① 用 dataset 跑前向 → 收集每层激活统计（给 modifier 决策用）
   ② 按 recipe 里 modifier 的【顺序】逐个应用：
        modifier 1  →  对模型做一步量化变换（如搜 scale）
        modifier 2  →  再做一步（如真正压 INT4）
        ...
   ③ 保存 → compressed-tensors 格式（config.json 里写 quantization_config）
        │
        ▼
  量化模型（可被 vLLM 加载）
```

**关键**：`recipe` 是一个**有序列表**——量化是分步的，顺序即流水线。比如 AWQ 是「先搜 scale、后压 INT4」两步，所以 recipe 有两个 modifier，顺序不能反。

## 2. 五大组件：是什么 / 为什么需要 / 什么格式

| 组件 | 是什么 | 为什么需要 | 什么格式 |
|---|---|---|---|
| **dataset** | 校准数据 | 量化算法要看**真实激活分布**（AWQ 挑显著通道、SmoothQuant 找离群点）才能定 scale | HuggingFace `Dataset`，含 `text` 列；oneshot 内部用 tokenizer 处理，你只给原始文本 |
| **recipe** | 量化步骤的有序清单 | 量化**分步**（先搜 scale 后量化），顺序即流水线 | `[modifier1, modifier2, ...]`（list） |
| **modifier** | 一个量化步骤的封装 | 每步做一件量化变换（搜 scale / 压位宽 / 平滑…），带参数控制 | `QuantizationModifier(...)` / `AWQModifier()` / `GPTQModifier(...)` 等类的实例 |
| **scheme** | 量化方案 | 决定**位宽 + 对称性 + 粒度**（per-channel/group） | 字符串：`W4A16_ASYM` / `W8A8` / `FP8_DYNAMIC` |
| **quantization_config** | 产物的量化元数据 | vLLM 据此**加载**量化模型；也是你验证「量化对没对」的依据 | 产物 `config.json` 里的一段（compressed-tensors 格式） |

> 一句话：你给 oneshot 一个 **model + dataset + recipe**（recipe 是 modifier 列表，每个 modifier 带 scheme），它吐出一个**带 quantization_config 的量化模型**。

## 3. 三种方法什么关系

三种方法都套上面那条通路，区别**只在 recipe 里塞了哪些 modifier + 用什么 scheme**。先看一张总表：

| 方法 | 权重 | 激活 | 数值格式 | 需补偿算法？ | recipe（modifier 组合） |
|---|---|---|---|---|---|
| **AWQ** (s2) | INT4 | FP16 | 整数 | 是（搜 scale 保护显著通道）| `AWQModifier()` + `QuantizationModifier(scheme="W4A16_ASYM")` |
| **SmoothQuant** (s3) | INT8 | INT8 | 整数 | 是（离群点迁到权重）| `SmoothQuantModifier()` + `GPTQModifier(scheme="W8A8")` |
| **FP8** (s1) | FP8 | FP8 | 浮点 | **否**（浮点大动态范围，直接 cast）| `QuantizationModifier(scheme="FP8_DYNAMIC")` |

**最该记住的洞察**（M1 讲过原理）：
- **INT 量化（AWQ/SmoothQuant）的死穴是「激活离群点让整数网格崩」**，所以它俩都得加一个 modifier 来补偿——AWQ 用 `AWQModifier` 搜 scale 保护显著通道，SmoothQuant 用 `SmoothQuantModifier` 把离群点迁到权重。这就是它俩 recipe 有**两个** modifier 的原因（先补偿、后压位宽）。
- **FP8 是浮点**（有指数位、动态范围远大于定点 INT），离群点不会让它崩，**直接 cast 就行、不需要补偿**——所以 FP8 的 recipe 只有**一个** modifier。

> 所以：**INT 要两步（补偿 + 量化），FP8 只要一步（直接 cast）**。这是三种方法在通路上最本质的区别。

In [ ]:
# 最小演示：用一个 tiny 内存模型 + 通用 W8A8 scheme，把上面那条通路真跑一遍（CPU 可跑，不依赖 FP8 硬件）。
# 目的不是产出可用模型，而是让你看到 "model + dataset + recipe → oneshot → 带 quantization_config 的产物" 这条数据流。
from transformers import Qwen2Config, Qwen2ForCausalLM, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from datasets import Dataset

# oneshot 要 tokenizer 处理校准文本——内存 tiny 模型没有，构造一个词表对齐的 word-level tokenizer
_tk = Tokenizer(WordLevel(vocab={str(i): i for i in range(320)}, unk_token="0"))
_tk.pre_tokenizer = Whitespace()
tiny_tok = PreTrainedTokenizerFast(tokenizer_object=_tk, unk_token="0", pad_token="0",
                                   eos_token="0", bos_token="0", model_max_length=64)

tiny = Qwen2ForCausalLM(Qwen2Config(
    num_hidden_layers=2, hidden_size=64, intermediate_size=128,
    num_attention_heads=4, num_key_value_heads=2, vocab_size=320,
    tie_word_embeddings=True)).eval()

calib = Dataset.from_dict({"text": [" ".join(str(i % 50) for i in range(40))] * 4})   # 占位校准文本（词表内 token）

recipe = [QuantizationModifier(scheme="W8A8", targets="Linear", ignore=["lm_head"])]  # 一个 modifier

oneshot(model=tiny, tokenizer=tiny_tok, dataset=calib, recipe=recipe, max_seq_length=32, num_calibration_samples=4)
demo_out = OUT_ROOT / "s0-demo-w8a8"
demo_out.mkdir(parents=True, exist_ok=True)
tiny.save_pretrained(demo_out)

qc = json.loads((demo_out / "config.json").read_text())["quantization_config"]
print("产物 quantization_config 的 quant_method:", qc.get("quant_method"))
print("通路跑通：model + dataset + recipe → oneshot → 带 quantization_config 的量化模型 ✓")

## 4. 理解检查（继续前自答）

读完上面，你应该能答出：
1. `oneshot(model, dataset, recipe)` 内部三步是什么？（前向收集统计 → 按 recipe 顺序应用 modifier → 保存）
2. 为什么 `recipe` 是个**列表**、且顺序重要？（量化分步，顺序即流水线）
3. 为什么 AWQ/SmoothQuant 的 recipe 有**两个** modifier，FP8 只有**一个**？（INT 要先补偿离群点再量化；FP8 浮点不需补偿、直接 cast）
4. 校准 `dataset` 为什么不能省？（量化算法要看真实激活分布定 scale）

答得出来再进 s1/s2/s3——它们只是在这条通路上换不同的 recipe。